# Prompt and Context Improvement for Dataset-Agnostic Turkish Legal RAG

This notebook improves the generation prompt and context formatting of the best pre-fine-tuning pipeline.

Best previous setup:
- Hybrid retrieval
- Turkish BGE reranker fusion
- Strict prompt
- Mistral-7B-Instruct

This notebook keeps the same retrieval and reranking setup, but improves:
- prompt structure
- answer format control
- context presentation
- dataset configurability

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
project_path = "/content/drive/MyDrive/turkish_legal_rag"

data_path = f"{project_path}/data"
processed_path = f"{project_path}/data/processed"
faiss_path = f"{project_path}/outputs/faiss"
metrics_path = f"{project_path}/outputs/metrics"

print("Project path:", project_path)
print("Processed path:", processed_path)
print("FAISS path:", faiss_path)
print("Metrics path:", metrics_path)

Project path: /content/drive/MyDrive/turkish_legal_rag
Processed path: /content/drive/MyDrive/turkish_legal_rag/data/processed
FAISS path: /content/drive/MyDrive/turkish_legal_rag/outputs/faiss
Metrics path: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics


In [3]:
DATASET_CONFIG = {
    "dataset_name": "current_turkish_legal_dataset",

    "corpus_path": f"{processed_path}/retrieval_corpus.csv",
    "qa_path": f"{processed_path}/test_qa.csv",

    "text_column": "chunk_text",
    "question_column": "question",
    "answer_column": "answer",

    "source_column": "source",
    "chunk_id_column": "chunk_id",

    "faiss_index_path": f"{faiss_path}/baseline_faiss.index"
}

DATASET_CONFIG

{'dataset_name': 'current_turkish_legal_dataset',
 'corpus_path': '/content/drive/MyDrive/turkish_legal_rag/data/processed/retrieval_corpus.csv',
 'qa_path': '/content/drive/MyDrive/turkish_legal_rag/data/processed/test_qa.csv',
 'text_column': 'chunk_text',
 'question_column': 'question',
 'answer_column': 'answer',
 'source_column': 'source',
 'chunk_id_column': 'chunk_id',
 'faiss_index_path': '/content/drive/MyDrive/turkish_legal_rag/outputs/faiss/baseline_faiss.index'}

In [4]:
!pip install -q -U faiss-cpu sentence-transformers rank-bm25 transformers accelerate bitsandbytes rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.7 MB/s eta 0:00:00


In [5]:
import os
import re
import json
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [6]:
def load_table(path):
    path = str(path)

    if path.endswith(".csv"):
        return pd.read_csv(path)
    elif path.endswith(".json"):
        return pd.read_json(path)
    elif path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    elif path.endswith(".pkl") or path.endswith(".pickle"):
        with open(path, "rb") as f:
            return pickle.load(f)
    else:
        raise ValueError(f"Unsupported file format: {path}")

In [7]:
chunks_df = load_table(DATASET_CONFIG["corpus_path"])
test_qa_df = load_table(DATASET_CONFIG["qa_path"])

print("Chunks shape:", chunks_df.shape)
print("QA shape:", test_qa_df.shape)

display(chunks_df.head())
display(test_qa_df.head())

Chunks shape: (3775, 5)
QA shape: (1500, 2)


,chunk_id,source_context_id,source,chunk_text,chunk_len
0,chunk_000000,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Türk Vatanı ve Milletinin ebedi varlığını ve Y...,263
1,chunk_000001,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,Dünya milletleri ailesinin eşit haklara sahip ...,194
2,chunk_000002,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Millet iradesinin mutlak üstünlüğü, egemenliği...",276
3,chunk_000003,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Kuvvetler ayrımının, Devlet organları arasında...",256
4,chunk_000004,kaggle_ctx_00000,Türkiye Cumhuriyeti Anayasası,"Hiçbir faaliyetin Türk milli menfaatlerinin, T...",370


,question,answer
0,Anayasanın 90. Maddesi Nasıl Uygulanır?,Milletlerarası antlaşmaların TBMM tarafından o...
1,Hukukta 'legitimate expectation' nedir?,"Legitimate expectation, bir kişinin belirli bi..."
2,"Anayasa madde 172'ye göre, devletin sanayi ve ...","Anayasa madde 172'ye göre, devlet, sanayi ve t..."
3,"Anayasa madde 158, uyuşmazlık mahkemesi'nin ku...","Anayasa madde 158'e göre, uyuşmazlık mahkemesi..."
4,"Bir grup avukat, Türkiye Büyük Millet Meclisi ...","Anayasanın 94. Maddesi, Türkiye Büyük Millet M..."


In [8]:
def validate_dataset_columns(chunks_df, qa_df, config):
    required_corpus_cols = [
        config["text_column"]
    ]

    required_qa_cols = [
        config["question_column"],
        config["answer_column"]
    ]

    missing_corpus = [
        col for col in required_corpus_cols
        if col not in chunks_df.columns
    ]

    missing_qa = [
        col for col in required_qa_cols
        if col not in qa_df.columns
    ]

    if missing_corpus:
        raise ValueError(f"Missing corpus columns: {missing_corpus}. Available: {list(chunks_df.columns)}")

    if missing_qa:
        raise ValueError(f"Missing QA columns: {missing_qa}. Available: {list(qa_df.columns)}")

    print("Dataset columns are valid.")


validate_dataset_columns(chunks_df, test_qa_df, DATASET_CONFIG)

Dataset columns are valid.


In [9]:
TEXT_COL = DATASET_CONFIG["text_column"]
QUESTION_COL = DATASET_CONFIG["question_column"]
ANSWER_COL = DATASET_CONFIG["answer_column"]

SOURCE_COL = DATASET_CONFIG.get("source_column", "source")
CHUNK_ID_COL = DATASET_CONFIG.get("chunk_id_column", "chunk_id")

if CHUNK_ID_COL not in chunks_df.columns:
    chunks_df[CHUNK_ID_COL] = [f"chunk_{i:06d}" for i in range(len(chunks_df))]

if SOURCE_COL not in chunks_df.columns:
    chunks_df[SOURCE_COL] = DATASET_CONFIG["dataset_name"]

print("Text column:", TEXT_COL)
print("Question column:", QUESTION_COL)
print("Answer column:", ANSWER_COL)
print("Source column:", SOURCE_COL)
print("Chunk ID column:", CHUNK_ID_COL)

Text column: chunk_text
Question column: question
Answer column: answer
Source column: source
Chunk ID column: chunk_id


In [10]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [11]:
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

embedding_model = SentenceTransformer(embedding_model_name)

index = faiss.read_index(DATASET_CONFIG["faiss_index_path"])

print("Embedding model loaded:", embedding_model_name)
print("FAISS vectors:", index.ntotal)
print("Chunks:", len(chunks_df))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
FAISS vectors: 3775
Chunks: 3775


In [12]:
if index.ntotal != len(chunks_df):
    print("WARNING: FAISS index size does not match corpus size.")
else:
    print("FAISS index and corpus size match.")

FAISS index and corpus size match.


In [13]:
def simple_turkish_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zçğıöşü0-9\s]", " ", text)
    tokens = text.split()

    stopwords = {
        "ve", "veya", "ile", "de", "da", "bir", "bu", "şu", "o",
        "için", "gibi", "olarak", "olan", "kadar", "ise", "ancak",
        "çok", "daha", "en", "mi", "mı", "mu", "mü"
    }

    return [t for t in tokens if t not in stopwords and len(t) > 1]

In [14]:
tokenized_corpus = [
    simple_turkish_tokenize(text)
    for text in chunks_df[TEXT_COL].astype(str).tolist()
]

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 ready.")

BM25 ready.


In [15]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=np.float32)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (scores.max() - scores.min())


def hybrid_retrieve_top_k(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    query_embedding = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(query_embedding)

    dense_scores, dense_indices = index.search(query_embedding, len(chunks_df))

    dense_scores = dense_scores[0]
    dense_indices = dense_indices[0]

    dense_score_map = {
        int(idx): float(score)
        for idx, score in zip(dense_indices, dense_scores)
    }

    dense_all_scores = np.array([
        dense_score_map.get(i, 0.0)
        for i in range(len(chunks_df))
    ])

    tokenized_query = simple_turkish_tokenize(query)
    bm25_scores = np.array(bm25.get_scores(tokenized_query))

    dense_norm = min_max_normalize(dense_all_scores)
    bm25_norm = min_max_normalize(bm25_scores)

    final_scores = alpha * dense_norm + (1 - alpha) * bm25_norm

    top_indices = np.argsort(final_scores)[::-1][:k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx][CHUNK_ID_COL],
            "source": chunks_df.iloc[idx][SOURCE_COL],
            "score": float(final_scores[idx]),
            "dense_score": float(dense_norm[idx]),
            "bm25_score": float(bm25_norm[idx]),
            "chunk_text": chunks_df.iloc[idx][TEXT_COL]
        })

    return results

In [16]:
USE_SOURCE_FILTER = True

In [17]:
def detect_source_filter(query):
    q = str(query).lower()

    if "anayasa" in q or "anayasanın" in q:
        return "Türkiye Cumhuriyeti Anayasası"

    return None

In [18]:
def hybrid_retrieve_top_k_filtered(query, model, index, chunks_df, bm25, k=5, alpha=0.5):
    source_filter = detect_source_filter(query) if USE_SOURCE_FILTER else None

    candidate_df = chunks_df.copy()

    if source_filter is not None and SOURCE_COL in candidate_df.columns:
        filtered_df = candidate_df[
            candidate_df[SOURCE_COL].astype(str).str.lower() == source_filter.lower()
        ].reset_index(drop=True)

        if len(filtered_df) > 0:
            candidate_df = filtered_df

    candidate_texts = candidate_df[TEXT_COL].astype(str).tolist()

    candidate_embeddings = embedding_model.encode(
        candidate_texts,
        convert_to_numpy=True,
        show_progress_bar=False
    ).astype("float32")

    faiss.normalize_L2(candidate_embeddings)

    temp_index = faiss.IndexFlatIP(candidate_embeddings.shape[1])
    temp_index.add(candidate_embeddings)

    tokenized_candidate_corpus = [
        simple_turkish_tokenize(text)
        for text in candidate_texts
    ]

    temp_bm25 = BM25Okapi(tokenized_candidate_corpus)

    return hybrid_retrieve_top_k(
        query,
        model,
        temp_index,
        candidate_df,
        temp_bm25,
        k=k,
        alpha=alpha
    )

In [19]:
turkish_reranker_model_name = "seroe/bge-reranker-v2-m3-turkish-triplet"

turkish_reranker = CrossEncoder(
    turkish_reranker_model_name,
    device="cpu",
    max_length=512
)

print("Turkish BGE reranker loaded:", turkish_reranker_model_name)

config.json:   0%|          | 0.00/884 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Turkish BGE reranker loaded: seroe/bge-reranker-v2-m3-turkish-triplet


In [20]:
def rerank_retrieved_chunks_turkish_bge(question, retrieved_results, top_k=None):
    pairs = [
        [question, item["chunk_text"]]
        for item in retrieved_results
    ]

    scores = turkish_reranker.predict(
        pairs,
        batch_size=4,
        show_progress_bar=False
    )

    scored_results = []

    for item, score in zip(retrieved_results, scores):
        scored_results.append({
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "original_rank": item["rank"],
            "original_hybrid_score": item["score"],
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "rerank_score": float(score),
            "chunk_text": item["chunk_text"]
        })

    scored_results = sorted(
        scored_results,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    if top_k is None:
        top_k = len(scored_results)

    final_results = []

    for new_rank, item in enumerate(scored_results[:top_k], start=1):
        item["rerank_rank"] = new_rank
        final_results.append(item)

    return final_results

In [21]:
def hybrid_retrieve_with_turkish_bge_fusion(
    query,
    model,
    index,
    chunks_df,
    bm25,
    candidate_k=10,
    final_k=3,
    alpha=0.5,
    hybrid_weight=0.7,
    rerank_weight=0.3
):
    candidates = hybrid_retrieve_top_k_filtered(
        query,
        model,
        index,
        chunks_df,
        bm25,
        k=candidate_k,
        alpha=alpha
    )

    reranked_all = rerank_retrieved_chunks_turkish_bge(
        question=query,
        retrieved_results=candidates,
        top_k=candidate_k
    )

    rerank_rank_map = {
        item["chunk_id"]: item["rerank_rank"]
        for item in reranked_all
    }

    rerank_score_map = {
        item["chunk_id"]: item["rerank_score"]
        for item in reranked_all
    }

    fused_results = []

    for item in candidates:
        chunk_id = item["chunk_id"]

        original_rank = item["rank"]
        rerank_rank = rerank_rank_map.get(chunk_id, candidate_k + 1)

        hybrid_rank_score = 1 / original_rank
        rerank_rank_score = 1 / rerank_rank

        fusion_score = (
            hybrid_weight * hybrid_rank_score
            + rerank_weight * rerank_rank_score
        )

        fused_results.append({
            "rank": None,
            "chunk_id": item["chunk_id"],
            "source": item["source"],
            "fusion_score": float(fusion_score),
            "original_rank": original_rank,
            "rerank_rank": rerank_rank,
            "rerank_score": float(rerank_score_map.get(chunk_id, 0.0)),
            "original_hybrid_score": item["score"],
            "dense_score": item["dense_score"],
            "bm25_score": item["bm25_score"],
            "chunk_text": item["chunk_text"]
        })

    fused_results = sorted(
        fused_results,
        key=lambda x: x["fusion_score"],
        reverse=True
    )

    final_results = []

    for new_rank, item in enumerate(fused_results[:final_k], start=1):
        item["rank"] = new_rank
        final_results.append(item)

    return final_results

In [31]:
def build_improved_legal_rag_prompt(question, retrieved_contexts):
    context_text = "\n\n".join([
        f"[Bağlam {i+1}]\n{ctx}"
        for i, ctx in enumerate(retrieved_contexts)
    ])

    prompt = f"""
Sen Türk hukuk metinleri üzerinde çalışan dikkatli bir RAG soru-cevap asistanısın.

Görevin:
Sadece verilen bağlamları kullanarak soruya kısa, net ve hukuki olarak doğru cevap vermek.

Zorunlu kurallar:
1. Cevabı yalnızca verilen bağlamlara dayandır.
2. Bağlamda açıkça bulunmayan bilgiyi uydurma.
3. Soru bir süre, tarih, sayı veya madde soruyorsa sadece ilgili değeri ve kısa açıklamasını ver.
4. Soru "aykırı mıdır", "çelişir mi", "uygun mudur" gibi bir değerlendirme soruyorsa cevaba mutlaka "Evet" veya "Hayır" ile başla.
5. Cevapta "Bağlam", "Context", "verilen metne göre" gibi ifadeler kullanma.
6. Cevap en fazla iki kısa cümle olmalı.
7. Alternatif cevap, yeni soru, örnek soru, başlık veya açıklama bölümü üretme.
8. Cevabı verdikten sonra dur.
9. Eğer bağlamda cevap yoksa sadece şunu yaz: "Verilen bağlamda bu sorunun cevabı bulunamamaktadır."

Bağlamlar:
{context_text}

Soru:
{question}

Kısa ve doğrudan cevap:
"""
    return prompt.strip()

In [32]:
from huggingface_hub import login
login()

In [33]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llm_model_name = "mistralai/Mistral-7B-Instruct-v0.2"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(llm_model_name)

model = AutoModelForCausalLM.from_pretrained(
    llm_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("LLM loaded:", llm_model_name)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

LLM loaded: mistralai/Mistral-7B-Instruct-v0.2


In [34]:
def generate_answer(prompt, max_new_tokens=80):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    markers = [
        "Kısa ve doğrudan cevap:",
        "Kısa cevap:",
        "Answer:"
    ]

    for marker in markers:
        if marker in generated_text:
            generated_text = generated_text.split(marker)[-1].strip()

    return generated_text.strip()

In [40]:
def clean_generated_answer(text):
    text = str(text).strip()

    # Prompt markers
    start_markers = [
        "Kısa ve doğrudan cevap:",
        "Kısa cevap:",
        "Answer:"
    ]

    for marker in start_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    # Stop if model starts generating extra sections/questions
    stop_markers = [
        "\nBaşka bir şekilde:",
        "\nCevap:",
        "\nSoru:",
        "\nQuestion:",
        "\nAlternatif",
        "\nAçıklama:",
        "\nDetaylı cevap:",
        "\nÖrnek:"
    ]

    for marker in stop_markers:
        if marker in text:
            text = text.split(marker)[0].strip()

    # Remove unwanted context labels
    unwanted = [
        "Context 1", "Context 2", "Context 3",
        "Bağlam 1", "Bağlam 2", "Bağlam 3"
    ]

    for u in unwanted:
        text = text.replace(u, "")

    # Remove empty parentheses like "()"
    text = re.sub(r"\(\s*\)", "", text)

    # Clean extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [45]:
def postprocess_answer_by_question_type(question, answer):
    q = str(question).lower()
    answer = str(answer).strip()

    short_answer_triggers = [
        "kaç",
        "ne zaman",
        "kime aittir",
        "en fazla kaç"
    ]

    if any(trigger in q for trigger in short_answer_triggers):
        sentences = re.split(r"(?<=[.!?])\s+", answer)
        if len(sentences) > 0:
            return sentences[0].strip()

    return answer

In [46]:
CANDIDATE_K = 10
FINAL_CONTEXT_K = 3
ALPHA = 0.5

HYBRID_WEIGHT = 0.7
RERANK_WEIGHT = 0.3

In [47]:
base_eval_questions = [
    {
        "question": "Egemenlik kime aittir?",
        "expected_answer": "Egemenlik kayıtsız şartsız Milletindir."
    },
    {
        "question": "Türkiye Cumhuriyetinin yönetim şekli nedir?",
        "expected_answer": "Türkiye Devleti bir Cumhuriyettir."
    },
    {
        "question": "Cumhurbaşkanının görev süresi kaç yıldır?",
        "expected_answer": "Cumhurbaşkanının görev süresi beş yıldır."
    },
    {
        "question": "Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?",
        "expected_answer": "Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir."
    }
]

base_eval_df = pd.DataFrame(base_eval_questions)

In [50]:
improved_prompt_small_results = []

for _, row in base_eval_df.iterrows():
    question = row["question"]
    expected_answer = row["expected_answer"]

    retrieved = hybrid_retrieve_with_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    improved_prompt_small_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_fusion_score": retrieved[0]["fusion_score"]
    })

improved_prompt_small_results_df = pd.DataFrame(improved_prompt_small_results)
improved_prompt_small_results_df

,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_original_rank,top1_rerank_rank,top1_fusion_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,1,1,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,1,3,0.8
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,1,1,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,1,1,1.0


In [51]:
for i, row in improved_prompt_small_results_df.iterrows():
    print("=" * 100)
    print("INDEX:", i)
    print("QUESTION:", row["question"])
    print("EXPECTED:", row["expected_answer"])
    print("GENERATED:", row["clean_generated_answer"])
    print("TOP1:", row["top1_chunk_id"], row["top1_source"])

INDEX: 0
QUESTION: Egemenlik kime aittir?
EXPECTED: Egemenlik kayıtsız şartsız Milletindir.
GENERATED: Egemenlik Türk Milleti'ne aittir.
TOP1: chunk_000269 Türkiye Cumhuriyeti Anayasası
INDEX: 1
QUESTION: Türkiye Cumhuriyetinin yönetim şekli nedir?
EXPECTED: Türkiye Devleti bir Cumhuriyettir.
GENERATED: Türkiye Cumhuriyetinin yönetim şekli bölünmez bir bütünlük olmasına dayalı bir halk ve toprak demokratik republiktir.
TOP1: chunk_000000 Türkiye Cumhuriyeti Anayasası
INDEX: 2
QUESTION: Cumhurbaşkanının görev süresi kaç yıldır?
EXPECTED: Cumhurbaşkanının görev süresi beş yıldır.
GENERATED: Cumhurbaşkanının görev süresi beş yıldır.
TOP1: chunk_000043 Türkiye Cumhuriyeti Anayasası
INDEX: 3
QUESTION: Bir kimse en fazla kaç defa Cumhurbaşkanı seçilebilir?
EXPECTED: Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
GENERATED: Bir kimse en fazla iki defa Cumhurbaşkanı seçilebilir.
TOP1: chunk_000043 Türkiye Cumhuriyeti Anayasası


In [52]:
manual_scores_improved_prompt_small = [
    1.0,
    0.0,
    1.0,
    1.0
]

improved_prompt_small_results_df["manual_score"] = manual_scores_improved_prompt_small

improved_prompt_small_score = improved_prompt_small_results_df["manual_score"].mean()

print("Improved Prompt 4-question score:", improved_prompt_small_score)
improved_prompt_small_results_df

Improved Prompt 4-question score: 0.75


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_original_rank,top1_rerank_rank,top1_fusion_score,manual_score
0,Egemenlik kime aittir?,Egemenlik kayıtsız şartsız Milletindir.,Egemenlik Türk Milleti'ne aittir. (Bağlam 1),Egemenlik Türk Milleti'ne aittir.,chunk_000269,Türkiye Cumhuriyeti Anayasası,1,1,1.0,1.0
1,Türkiye Cumhuriyetinin yönetim şekli nedir?,Türkiye Devleti bir Cumhuriyettir.,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,Türkiye Cumhuriyetinin yönetim şekli bölünmez ...,chunk_000000,Türkiye Cumhuriyeti Anayasası,1,3,0.8,0.0
2,Cumhurbaşkanının görev süresi kaç yıldır?,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,Cumhurbaşkanının görev süresi beş yıldır.,chunk_000043,Türkiye Cumhuriyeti Anayasası,1,1,1.0,1.0
3,Bir kimse en fazla kaç defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,Bir kimse en fazla iki defa Cumhurbaşkanı seçi...,chunk_000043,Türkiye Cumhuriyeti Anayasası,1,1,1.0,1.0


In [53]:
improved_prompt_small_results_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_4question_results.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Improved Prompt + Turkish BGE Reranker Fusion - 4 Question Sanity Evaluation",
    "manual_accuracy": improved_prompt_small_score,
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_4question_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Improved prompt 4-question results saved.")

Improved prompt 4-question results saved.


In [54]:
test_eval_df = test_qa_df.sample(n=20, random_state=42).reset_index(drop=True)

print(test_eval_df.shape)
test_eval_df.head()

(20, 2)


,question,answer
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...


In [55]:
improved_prompt_test_results = []

for _, row in tqdm(test_eval_df.iterrows(), total=len(test_eval_df)):
    question = row[QUESTION_COL]
    expected_answer = row[ANSWER_COL]

    retrieved = hybrid_retrieve_with_turkish_bge_fusion(
        question,
        embedding_model,
        index,
        chunks_df,
        bm25,
        candidate_k=CANDIDATE_K,
        final_k=FINAL_CONTEXT_K,
        alpha=ALPHA,
        hybrid_weight=HYBRID_WEIGHT,
        rerank_weight=RERANK_WEIGHT
    )

    contexts = [r["chunk_text"] for r in retrieved]

    prompt = build_improved_legal_rag_prompt(question, contexts)

    generated_answer = generate_answer(prompt)
    clean_answer = clean_generated_answer(generated_answer)
    clean_answer = postprocess_answer_by_question_type(question, clean_answer)

    improved_prompt_test_results.append({
        "question": question,
        "expected_answer": expected_answer,
        "generated_answer": generated_answer,
        "clean_generated_answer": clean_answer,
        "top1_chunk_id": retrieved[0]["chunk_id"],
        "top1_source": retrieved[0]["source"],
        "top1_context": retrieved[0]["chunk_text"],
        "top1_original_rank": retrieved[0]["original_rank"],
        "top1_rerank_rank": retrieved[0]["rerank_rank"],
        "top1_rerank_score": retrieved[0]["rerank_score"],
        "top1_fusion_score": retrieved[0]["fusion_score"],
        "retrieved_contexts": "\n\n".join(contexts)
    })

improved_prompt_test_results_df = pd.DataFrame(improved_prompt_test_results)
improved_prompt_test_results_df.head()

100%|██████████| 20/20 [06:20<00:00, 19.02s/it]


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 17/5/1987 tarihinde eklendi.,Geçici madde 20 17/5/1987 tarihinde eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde içerikleri ulaşım araçları...","Evet, TCK 121 madde içerikleri ulaşım araçları...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...


In [56]:
improved_prompt_test_results_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_rag_testset_generation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Improved prompt test generation results saved.")

Improved prompt test generation results saved.


In [57]:
for i, row in improved_prompt_test_results_df.iterrows():
    print("=" * 120)
    print("INDEX:", i)

    print("\nQUESTION:")
    print(row["question"])

    print("\nEXPECTED:")
    print(row["expected_answer"])

    print("\nCLEAN GENERATED:")
    print(row["clean_generated_answer"][:1000])

    print("\nTOP1 CHUNK ID:", row["top1_chunk_id"])
    print("TOP1 SOURCE:", row["top1_source"])
    print("TOP1 ORIGINAL RANK:", row["top1_original_rank"])
    print("TOP1 RERANK RANK:", row["top1_rerank_rank"])
    print("TOP1 RERANK SCORE:", row["top1_rerank_score"])
    print("TOP1 FUSION SCORE:", row["top1_fusion_score"])

INDEX: 0

QUESTION:
Anayasanın 101. Maddesiyle İlgili Tartışmalar Nelerdir?

EXPECTED:
Cumhurbaşkanının seçilme şartlarının sınırları ve uygulanması üzerine tartışmalar olabilir.

CLEAN GENERATED:
Anayasanın 101. Maddesiyle ilgili tartışmalar, gerekli çoğunluğun sağlanamaması halinde ikinci oylama yapılması ve seçimlerin geriye bırakılması ve ara seçimler h

TOP1 CHUNK ID: chunk_000210
TOP1 SOURCE: Türkiye Cumhuriyeti Anayasası
TOP1 ORIGINAL RANK: 1
TOP1 RERANK RANK: 4
TOP1 RERANK SCORE: 0.007053057197481394
TOP1 FUSION SCORE: 0.7749999999999999
INDEX: 1

QUESTION:
Bir grup vatandaş, belirli bir etnik grubun diğerlerinden daha fazla hakka sahip olması için imza kampanyası başlatmıştır. Bu durum Anayasanın 10. Maddesi ile nasıl çelişir?

EXPECTED:
Anayasanın 10. Maddesi, herkesin kanun önünde eşit olduğunu belirtir. Bu tür bir imza kampanyası Anayasa'ya aykırıdır.

CLEAN GENERATED:
Anayasanın 10. Maddesi, herkesin dil, ırk, renk, cinsiyet, siyasi düşünce, felsefi inanç, din, mezhep ve b

In [58]:
improved_prompt_test_results_df["is_valid_sample"] = True

# Önceki deneylerle aynı setup:
# index 19 invalid sample olarak skora dahil edilmiyor.
improved_prompt_test_results_df.loc[19, "is_valid_sample"] = False

manual_scores_improved_prompt = [
    0.0,
    0.5,
    0.5,
    0.0,
    0.5,
    0.0,
    0.0,
    1.0,
    0.5,
    0.0,
    1.0,
    0.0,
    0.5,
    1.0,
    0.5,
    0.5,
    0.5,
    0.0,
    0.5,
    0.0
]

improved_prompt_test_results_df["manual_score"] = manual_scores_improved_prompt

valid_improved_prompt_df = improved_prompt_test_results_df[
    improved_prompt_test_results_df["is_valid_sample"] == True
]

improved_prompt_test_score = valid_improved_prompt_df["manual_score"].mean()

print("Improved Prompt + Turkish BGE Reranker Fusion RAG Test Score:", improved_prompt_test_score)
print("Valid sample count:", len(valid_improved_prompt_df))
print("Total sample count:", len(improved_prompt_test_results_df))

Improved Prompt + Turkish BGE Reranker Fusion RAG Test Score: 0.39473684210526316
Valid sample count: 19
Total sample count: 20


In [59]:
improved_prompt_test_results_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_rag_testset_scored.csv",
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([{
    "method": "Improved Prompt + Turkish BGE Reranker Fusion RAG",
    "manual_accuracy": improved_prompt_test_score,
    "valid_sample_count": len(valid_improved_prompt_df),
    "total_sample_count": len(improved_prompt_test_results_df),
    "candidate_k": CANDIDATE_K,
    "final_context_k": FINAL_CONTEXT_K,
    "alpha": ALPHA,
    "hybrid_weight": HYBRID_WEIGHT,
    "rerank_weight": RERANK_WEIGHT
}]).to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_rag_testset_score.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Improved prompt scored results saved.")

Improved prompt scored results saved.


In [60]:
test_comparison_df = pd.DataFrame([
    {
        "method": "Base RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Base prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "Strict Prompt RAG",
        "retrieval_setup": "Hybrid retrieval top-5, context top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.263158
    },
    {
        "method": "FlashRank Fusion Reranker RAG",
        "retrieval_setup": "Hybrid top-10 + FlashRank reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.184211
    },
    {
        "method": "Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Strict prompt",
        "manual_accuracy": 0.315789
    },
    {
        "method": "Improved Prompt + Turkish BGE Reranker Fusion RAG",
        "retrieval_setup": "Hybrid top-10 + Turkish BGE reranker + rank fusion top-3",
        "prompt": "Improved legal prompt",
        "manual_accuracy": improved_prompt_test_score
    }
])

test_comparison_df

,method,retrieval_setup,prompt,manual_accuracy
0,Base RAG,"Hybrid retrieval top-5, context top-3",Base prompt,0.263158
1,Strict Prompt RAG,"Hybrid retrieval top-5, context top-3",Strict prompt,0.263158
2,FlashRank Fusion Reranker RAG,Hybrid top-10 + FlashRank reranker + rank fusi...,Strict prompt,0.184211
3,Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Strict prompt,0.315789
4,Improved Prompt + Turkish BGE Reranker Fusion RAG,Hybrid top-10 + Turkish BGE reranker + rank fu...,Improved legal prompt,0.394737


In [61]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_prompt_reranker_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Prompt and reranker comparison saved.")

Prompt and reranker comparison saved.


## Error Analysis for Improved Prompt Results

In [62]:
df_improved = improved_prompt_test_results_df.copy()

valid_improved_df = df_improved[df_improved["is_valid_sample"] == True].copy()

print("Valid sample count:", len(valid_improved_df))
print("Mean score:", valid_improved_df["manual_score"].mean())
print(valid_improved_df["manual_score"].value_counts())

Valid sample count: 19
Mean score: 0.39473684210526316
manual_score
0.5    9
0.0    7
1.0    3
Name: count, dtype: int64


In [63]:
retrieval_quality_labels_improved = [
    "partial_context",     # 0 - Article 101 geldi ama beklenen seçilme şartı/tartışma yönü tam değil.
    "correct_context",     # 1 - Article 10/eşitlik bağlamı var ama cevap eksik/kesilmiş.
    "correct_context",     # 2 - Article 17 bağlamı var ama hukuki sonuç ters.
    "wrong_context",       # 3 - Geçici madde 20 tarihi yanlış bağlamdan geliyor.
    "wrong_context",       # 4 - TCK 121 yerine yanlış/Anayasa 121 bağlamı.
    "wrong_context",       # 5 - Cumhurbaşkanı yemini yerine alakasız yemin/yazman cevabı.
    "wrong_context",       # 6 - Article 122 cevabı yanlış.
    "correct_context",     # 7 - Yedi gün doğru context.
    "correct_context",     # 8 - Article 50 bağlamı var ama açıklama karışık.
    "correct_context",     # 9 - Article 108 bağlamı var ama Silahlı Kuvvetler istisnası yanlış.
    "correct_context",     # 10 - Article 13 bağlamı doğru ve cevap doğru.
    "wrong_context",       # 11 - KVKK ilgili kişi yerine yanlış bağlam.
    "partial_context",     # 12 - AYM kesinlik sorusu kısmen yakalanmış ama tam bağlam değil.
    "correct_context",     # 13 - Article 63 doğru.
    "partial_context",     # 14 - Article 47 kısmen doğru bağlam ama eksik.
    "partial_context",     # 15 - Kişisel veriler için genel kanun bağlamı var ama KVKK net değil.
    "correct_context",     # 16 - Article 140 doğru bağlam, cevap eksik.
    "wrong_context",       # 17 - Ekonomik ve Sosyal Konsey için yanlış madde.
    "partial_context",     # 18 - Bilgi Edinme Kurulu bağlamı kısmen var.
    "dataset_mismatch"     # 19 - invalid sample.
]

generation_quality_labels_improved = [
    "wrong_generation",    # 0
    "partial_generation",  # 1
    "wrong_generation",    # 2
    "wrong_generation",    # 3
    "partial_generation",  # 4
    "wrong_generation",    # 5
    "wrong_generation",    # 6
    "correct_generation",  # 7
    "partial_generation",  # 8
    "wrong_generation",    # 9
    "correct_generation",  # 10
    "wrong_generation",    # 11
    "partial_generation",  # 12
    "correct_generation",  # 13
    "partial_generation",  # 14
    "partial_generation",  # 15
    "partial_generation",  # 16
    "wrong_generation",    # 17
    "partial_generation",  # 18
    "not_applicable"       # 19
]

error_type_labels_improved = [
    "partial_context",                    # 0
    "incomplete_answer",                  # 1
    "correct_context_wrong_generation",   # 2
    "wrong_context",                      # 3
    "wrong_context_partial_answer",       # 4
    "wrong_context",                      # 5
    "wrong_context",                      # 6
    "correct",                            # 7
    "partial_answer",                     # 8
    "correct_context_wrong_generation",   # 9
    "correct",                            # 10
    "wrong_context",                      # 11
    "partial_answer",                     # 12
    "correct",                            # 13
    "partial_answer",                     # 14
    "partial_answer",                     # 15
    "partial_answer",                     # 16
    "wrong_context",                      # 17
    "partial_answer",                     # 18
    "dataset_mismatch"                    # 19
]

notes_improved = [
    "Expected discussion about Article 101 eligibility/application, but generated answer focused on election procedure.",
    "Equality principle is mentioned, but answer is incomplete and does not clearly state unconstitutionality.",
    "Relevant Article 17 context exists, but the generated legal conclusion starts with 'Hayır' and contradicts expected answer.",
    "Expected date is 20 May 2016, generated date is 17/5/1987.",
    "Generated answer partially says violation is not fixed, but context and reasoning are wrong/incomplete.",
    "Wrong context; answer confuses presidential oath with an unrelated oath/writer issue.",
    "Generated says Article 122 does not exist, which is incorrect.",
    "Correctly answers seven days.",
    "Generated starts with correct direction but explanation is legally confused/incomplete.",
    "Expected DDK cannot inspect Armed Forces, but generated answer does not state this exception.",
    "Correctly states that arbitrary limitation of fundamental rights is unconstitutional and must be based on law.",
    "Wrong context; expected KVKK definition of data subject.",
    "Partially captures finality/enforcement idea, but does not explain that decisions are final and cannot be appealed.",
    "Correct answer for Article 63.",
    "Partially discusses Article 47/state nationalization but incomplete for the expected answer.",
    "Partially correct: personal data is processed by law, but KVKK and other laws are not explicitly mentioned.",
    "Correct main idea that judges' and prosecutors' duties/powers are regulated by law, but details are missing.",
    "Wrong article number; expected Article 115, generated Article 109.",
    "Mentions Information Evaluation Board but answer is incomplete and cut.",
    "Invalid sample: question and expected answer are mismatched."
]

df_improved["retrieval_quality"] = retrieval_quality_labels_improved
df_improved["generation_quality"] = generation_quality_labels_improved
df_improved["error_type"] = error_type_labels_improved
df_improved["notes"] = notes_improved

df_improved.head()

,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_context,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score,retrieval_quality,generation_quality,error_type,notes
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000210,Türkiye Cumhuriyeti Anayasası,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,1,4,0.007053,0.7750,Madde 158 – Uyuşmazlık Mahkemesi adli ve idari...,True,0.0,partial_context,wrong_generation,partial_context,Expected discussion about Article 101 eligibil...
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000188,Türkiye Cumhuriyeti Anayasası,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",1,4,0.000305,0.7750,"Madde 150 – Kanunların, Cumhurbaşkanlığı karar...",True,0.5,correct_context,partial_generation,incomplete_answer,"Equality principle is mentioned, but answer is..."
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1,1,0.385170,1.0000,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5,correct_context,wrong_generation,correct_context_wrong_generation,"Relevant Article 17 context exists, but the ge..."
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 17/5/1987 tarihinde eklendi.,Geçici madde 20 17/5/1987 tarihinde eklendi.,chunk_000600,Bilgi Edinme Kanunu,Madde 20- Açıklanması veya zamanından önce açı...,1,8,0.000864,0.7375,Madde 20- Açıklanması veya zamanından önce açı...,True,0.0,wrong_context,wrong_generation,wrong_context,"Expected date is 20 May 2016, generated date i..."
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde içerikleri ulaşım araçları...","Evet, TCK 121 madde içerikleri ulaşım araçları...",chunk_000101,Türkiye Cumhuriyeti Anayasası,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,1,6,0.000157,0.7500,Madde 121 – (Mülga: 21/1/2017-6771/16 md.) B. ...,True,0.5,wrong_context,partial_generation,wrong_context_partial_answer,Generated answer partially says violation is n...


In [64]:
valid_improved_df = df_improved[df_improved["is_valid_sample"] == True].copy()

error_summary_improved_df = valid_improved_df["error_type"].value_counts().reset_index()
error_summary_improved_df.columns = ["error_type", "count"]
error_summary_improved_df["percentage"] = (
    error_summary_improved_df["count"] / len(valid_improved_df) * 100
).round(2)

error_summary_improved_df

,error_type,count,percentage
0,partial_answer,6,31.58
1,wrong_context,5,26.32
2,correct,3,15.79
3,correct_context_wrong_generation,2,10.53
4,partial_context,1,5.26
5,incomplete_answer,1,5.26
6,wrong_context_partial_answer,1,5.26


In [65]:
retrieval_summary_improved_df = valid_improved_df["retrieval_quality"].value_counts().reset_index()
retrieval_summary_improved_df.columns = ["retrieval_quality", "count"]
retrieval_summary_improved_df["percentage"] = (
    retrieval_summary_improved_df["count"] / len(valid_improved_df) * 100
).round(2)

retrieval_summary_improved_df

,retrieval_quality,count,percentage
0,correct_context,8,42.11
1,wrong_context,6,31.58
2,partial_context,5,26.32


In [66]:
generation_summary_improved_df = valid_improved_df["generation_quality"].value_counts().reset_index()
generation_summary_improved_df.columns = ["generation_quality", "count"]
generation_summary_improved_df["percentage"] = (
    generation_summary_improved_df["count"] / len(valid_improved_df) * 100
).round(2)

generation_summary_improved_df

,generation_quality,count,percentage
0,wrong_generation,8,42.11
1,partial_generation,8,42.11
2,correct_generation,3,15.79


In [67]:
valid_improved_df.groupby("error_type")["manual_score"].agg(["count", "mean"])

,count,mean
error_type,,
correct,3,1.00
correct_context_wrong_generation,2,0.25
incomplete_answer,1,0.50
partial_answer,6,0.50
partial_context,1,0.00
wrong_context,5,0.00
wrong_context_partial_answer,1,0.50


In [68]:
valid_improved_df.groupby("retrieval_quality")["manual_score"].agg(["count", "mean"])

,count,mean
retrieval_quality,,
correct_context,8,0.625000
partial_context,5,0.400000
wrong_context,6,0.083333


In [69]:
df_improved.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_error_analysis.csv",
    index=False,
    encoding="utf-8-sig"
)

error_summary_improved_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_error_type_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

retrieval_summary_improved_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_retrieval_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

generation_summary_improved_df.to_csv(
    f"{metrics_path}/improved_prompt_turkish_bge_generation_quality_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Improved prompt error analysis files saved.")

Improved prompt error analysis files saved.


In [70]:
test_comparison_df.to_csv(
    f"{metrics_path}/rag_prompt_reranker_comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Prompt and reranker comparison saved.")

Prompt and reranker comparison saved.
